1.具有单隐藏层的多层感知机

In [2]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size=(2, 4))
net(X)

tensor([[-0.3570],
        [-0.4081]], grad_fn=<AddmmBackward0>)

2.参数访问 

拿出每一层的权重

net[0]:nn.Linear(4, 8)
net[1]:nn.ReLU()
net[2]:nn.Linear(8, 1)

In [3]:
print(net[2].state_dict())

OrderedDict([('weight', tensor([[-0.1366,  0.2724,  0.0867, -0.1450, -0.0249,  0.2374,  0.2053,  0.0230]])), ('bias', tensor([-0.3344]))])


2.1 目标参数

访问其偏置 和 梯度

In [4]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)
net[2].weight.grad == None

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([-0.3344], requires_grad=True)
tensor([-0.3344])


True

一次性访问所有参数

In [5]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])

# 整个网络的parameter拿出
print(*[(name, param.shape) for name, param in net.named_parameters()])

# 拿到第三层的偏移
net.state_dict()['2.bias'].data

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


tensor([-0.3344])

3. 从嵌套块收集参数

In [7]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

# 嵌套了 4个blcok1
def block2():
    net = nn.Sequential()
    
    # 方便可视化
    for i in range(4):
        net.add_module(f'block{i}',block1())
    
    return net


# 再外面再套一层
rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

tensor([[-0.0281],
        [-0.0278]], grad_fn=<AddmmBackward0>)

看下刚才设计的网络

In [8]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


4. 初始化参数

内置初始化

(normal/函数名)_ 表示会原地赋值，不会返回

In [11]:
# m是传入的模组
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)
        
net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([ 5.4457e-03, -1.1700e-02, -4.9985e-03,  1.4539e-05]), tensor(0.))

In [12]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)
        
net.apply(init_constant)
# net[0] 是4x8 
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

5 对于某些块应用不同的初始化方法

In [14]:
def xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
        
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([-0.1131, -0.0572,  0.3825, -0.0490])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


6 自定义初始化

In [15]:
def  my_init(m):
    if type(m) == nn.Linear:
        
        # 打印一下初始化信息
        print("Init",
             *[(name, param.shape) for name, param in m.named_parameters()][0])

        nn.init.uniform_(m.weight, -10, 10)
        # 保留 绝对值>=5 的权重，其余设为0
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[ 9.1935, -8.6983,  7.4510,  6.1950],
        [-8.0330,  0.0000,  0.0000,  0.0000]], grad_fn=<SliceBackward0>)

7 参数绑定

In [16]:
shared = nn.Linear(8, 8)

# 第2个和第4个层是一样的 改一个另一个也会改
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(), shared,
                    nn.ReLU(), nn.Linear(8, 1))
net(X)

tensor([[0.1767],
        [0.1783]], grad_fn=<AddmmBackward0>)